# InstantID Mobile — Full Pipeline (Run.ai H200)

Export InstantID → ONNX → INT8 trên H200 GPU.

| Bước | Cell | Thời gian |
|------|------|-----------|
| 0. Setup venv + install | 0 | ~3 phút |
| 1. Clone + download models | 1 | ~5 phút |
| 2. Fuse LCM-LoRA | 2 | ~2 phút |
| 3. Export ONNX | 3 | ~15 phút |
| 4. Quantize INT8 | 4 | ~10 phút |
| 5. Test | 5 | ~2 phút |
| 6. Save local | 6 | ~1 phút |

**H200:** 80GB VRAM + 200GB+ RAM → UNet fully quantized, no workarounds.

**Output:** `/workspace/Instantid-mobile/onnx-int8/` (~2.5 GB total)

**Run All** → chạy thẳng từ đầu đến cuối, không cần bấm từng cell.

---
## Cell 0 — Setup venv + Install Dependencies

In [ ]:
import subprocess, sys, os

WORKDIR = '/workspace' if os.path.exists('/workspace') else '/tmp'
VENV_DIR = f'{WORKDIR}/venv-instantid'
VENV_PIP = f'{VENV_DIR}/bin/pip'
VENV_PYTHON = f'{VENV_DIR}/bin/python'

# Tạo venv nếu chưa có
if not os.path.exists(VENV_PYTHON):
    print(f'Creating venv at {VENV_DIR}...')
    subprocess.check_call([sys.executable, '-m', 'venv', VENV_DIR])
    print('venv created')
else:
    print(f'venv already exists at {VENV_DIR}')

# Install dependencies vào venv
print('Installing dependencies into venv...')
subprocess.check_call([VENV_PIP, 'install', '-q', '--upgrade', 'pip'])
subprocess.check_call([VENV_PIP, 'install', '-q',
    'torch', 'torchvision', '--index-url', 'https://download.pytorch.org/whl/cu121'])
subprocess.check_call([VENV_PIP, 'install', '-q',
    'diffusers>=0.28.0', 'transformers>=4.40.0', 'huggingface_hub>=0.23.0',
    'accelerate>=0.30.0', 'safetensors', 'peft',
    'onnx', 'onnxscript', 'onnxruntime', 'onnxslim',
    'insightface', 'opencv-python', 'Pillow', 'scipy', 'psutil'])

# Verify
result = subprocess.run([VENV_PYTHON, '-c', '''
import torch, diffusers, onnxruntime, psutil
print(f"PyTorch {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
if torch.cuda.is_available():
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.0f} GB")
print(f"System RAM: {psutil.virtual_memory().total / 1024**3:.0f} GB")
print(f"diffusers {diffusers.__version__}")
print(f"onnxruntime {onnxruntime.__version__}")
'''], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)

print(f'\nvenv Python: {VENV_PYTHON}')
!df -h {WORKDIR} 2>/dev/null || df -h / | tail -1

In [ ]:
# Helper: chạy script trong venv
import subprocess, os

WORKDIR = '/workspace' if os.path.exists('/workspace') else '/tmp'
VENV_PYTHON = f'{WORKDIR}/venv-instantid/bin/python'
PROJECT_DIR = f'{WORKDIR}/Instantid-mobile'

def run(cmd, check=True):
    """Run a command using venv python, in project dir."""
    full_cmd = cmd.replace('python ', f'{VENV_PYTHON} ')
    print(f'$ {full_cmd}')
    result = subprocess.run(
        full_cmd, shell=True, cwd=PROJECT_DIR,
        capture_output=False
    )
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed with code {result.returncode}')
    return result

print(f'Project dir: {PROJECT_DIR}')
print(f'venv python: {VENV_PYTHON}')
print('Helper loaded')

---
## Cell 1 — Clone Repo + Download Models

In [ ]:
import os
os.chdir(WORKDIR)

if not os.path.exists('Instantid-mobile/.git'):
    !git clone https://github.com/Hert4/Instantid-mobile
else:
    !cd Instantid-mobile && git pull origin main

os.chdir(PROJECT_DIR)
print(f'Working dir: {os.getcwd()}')

In [ ]:
# Download ALL models — lưu local trong project dir
run(f'''python -c "
import os
os.makedirs('checkpoints/ControlNetModel', exist_ok=True)
os.makedirs('models/antelopev2', exist_ok=True)
from huggingface_hub import hf_hub_download, snapshot_download

# IP-Adapter
if not os.path.exists('checkpoints/ip-adapter.bin'):
    print('Downloading IP-Adapter...')
    hf_hub_download('InstantX/InstantID', 'ip-adapter.bin', local_dir='./checkpoints')
print('IP-Adapter OK')

# ControlNet
if not os.path.exists('checkpoints/ControlNetModel/config.json'):
    print('Downloading ControlNet...')
    snapshot_download('InstantX/InstantID', allow_patterns=['ControlNetModel/*'], local_dir='./checkpoints')
print('ControlNet OK')

# LCM-LoRA
if not os.path.exists('checkpoints/pytorch_lora_weights.safetensors'):
    print('Downloading LCM-LoRA...')
    hf_hub_download('latent-consistency/lcm-lora-sdxl', 'pytorch_lora_weights.safetensors', local_dir='./checkpoints')
print('LCM-LoRA OK')

# InsightFace
import glob
if not glob.glob('models/**/scrfd_10g_bnkps.onnx', recursive=True):
    print('Downloading InsightFace...')
    try:
        from insightface.app import FaceAnalysis
        app = FaceAnalysis(name='antelopev2', root='./models', providers=['CPUExecutionProvider'])
        app.prepare(ctx_id=-1, det_size=(640, 640))
    except:
        for f in ['scrfd_10g_bnkps.onnx', 'glintr100.onnx']:
            hf_hub_download('DIAMONIK7777/antelopev2', f, local_dir='./models/antelopev2')
print('InsightFace OK')

# SDXL base
if not os.path.exists('sdxl-base/model_index.json') and not os.path.exists('sdxl-base-lcm/model_index.json'):
    print('Downloading SDXL base (~6.5 GB)...')
    snapshot_download('stabilityai/stable-diffusion-xl-base-1.0', local_dir='./sdxl-base',
        ignore_patterns=['*.bin','*.ckpt','flax_model*','tf_model*','*.onnx*','*.msgpack','*.xml','*.pb'])
print('SDXL base OK')
print('All models downloaded')
"''')

---
## Cell 2 — Fuse LCM-LoRA

In [ ]:
import os, shutil
os.chdir(PROJECT_DIR)

if os.path.exists('sdxl-base-lcm/model_index.json'):
    print('sdxl-base-lcm already exists — skip fuse')
else:
    run('python scripts/fuse_lcm_lora.py --device cuda --delete_source')
    assert os.path.exists('sdxl-base-lcm/model_index.json'), 'Fuse FAILED!'

# Cleanup SDXL base gốc
if os.path.exists('sdxl-base'):
    shutil.rmtree('sdxl-base', ignore_errors=True)
    print('Freed sdxl-base')

print('Fuse complete')

---
## Cell 3 — Export ALL ONNX

In [ ]:
import os
os.chdir(PROJECT_DIR)
os.makedirs('onnx', exist_ok=True)

for comp in ['ip_adapter', 'controlnet', 'text_encoders', 'vae', 'unet']:
    print(f'\n{"="*60}')
    print(f'EXPORTING: {comp}')
    print(f'{"="*60}')
    run(f'python scripts/export_all.py --only {comp}')

# Summary
print(f'\n{"="*60}')
print('ONNX EXPORT SUMMARY')
print(f'{"="*60}')
total_mb = 0
for d in sorted(os.listdir('onnx')):
    p = os.path.join('onnx', d)
    if os.path.isdir(p):
        mb = sum(os.path.getsize(os.path.join(p, f)) for f in os.listdir(p)
                 if os.path.isfile(os.path.join(p, f))) / 1024**2
        total_mb += mb
        print(f'  {d:25s} {mb:>8.0f} MB')
print(f'  {"TOTAL":25s} {total_mb:>8.0f} MB ({total_mb/1024:.1f} GB)')
assert total_mb > 5000, f'Export incomplete! {total_mb:.0f} MB (expected ~11 GB)'

---
## Cell 4 — Quantize ALL to INT8

H200 has 200GB+ RAM → UNet fully quantized (impossible on Colab 12GB).

In [ ]:
import os, shutil
os.chdir(PROJECT_DIR)

# Free disk — sdxl-base-lcm không cần nữa sau export
shutil.rmtree('sdxl-base-lcm', ignore_errors=True)

# Quantize từng component
for comp in ['text_encoder', 'ip_adapter', 'vae', 'text_encoder_2', 'controlnet', 'unet']:
    print(f'\n{"="*60}')
    print(f'QUANTIZING: {comp}')
    print(f'{"="*60}')
    run(f'python scripts/quantize_all.py --skip_slim --only {comp}')

# Summary
print(f'\n{"="*60}')
print('INT8 QUANTIZATION SUMMARY')
print(f'{"="*60}')
total_mb = 0
for root, _, files in os.walk('onnx-int8'):
    for f in sorted(files):
        if f.endswith(('.onnx', '.pb', '.onnx_data')):
            p = os.path.join(root, f)
            mb = os.path.getsize(p) / 1024**2
            total_mb += mb
            rel = os.path.relpath(p, 'onnx-int8')
            print(f'  {rel:50s} {mb:>8.0f} MB')
print(f'  {"TOTAL":50s} {total_mb:>8.0f} MB ({total_mb/1024:.1f} GB)')

# Xóa FP16 originals
if total_mb > 500:
    shutil.rmtree('onnx', ignore_errors=True)
    print('\nFreed onnx/ (FP16 originals)')
else:
    print('\nQuantize output too small — keeping onnx/')

---
## Cell 5 — Test Pipeline

In [ ]:
import os
os.chdir(PROJECT_DIR)
os.makedirs('test_images', exist_ok=True)

# ══ Đặt ảnh face vào đây ══
# Copy từ persistent volume hoặc upload:
# !cp /workspace/data/my_face.jpg test_images/face.jpg

FACE_IMAGE = 'test_images/face.jpg'
if not os.path.exists(FACE_IMAGE):
    print('No face image found at test_images/face.jpg')
    print('Copy your photo: !cp /path/to/face.jpg test_images/face.jpg')
    print('Then re-run this cell.')
else:
    run(f'''python scripts/test_onnx_pipeline.py \
        --face_image "{FACE_IMAGE}" \
        --onnx_dir ./onnx-int8 \
        --insightface_dir ./models/antelopev2 \
        --prompt "portrait photo of a person, professional lighting, 4k, detailed" \
        --steps 4 --seed 42 \
        --output ./output/test_result.png \
        --verbose''')

    if os.path.exists('./output/test_result.png'):
        try:
            from IPython.display import Image, display
            display(Image('./output/test_result.png', width=512))
        except:
            pass
        print('Pipeline test PASSED!')
    else:
        print('Output not found — check errors above')

---
## Cell 6 — Save Results (Local)

In [ ]:
import os
os.chdir(PROJECT_DIR)

# Final output summary
print('=== Final Output (local) ===')
print(f'Location: {PROJECT_DIR}/onnx-int8/')
print()
total = 0
for root, _, files in os.walk('onnx-int8'):
    for f in sorted(files):
        if f.endswith(('.onnx', '.pb', '.onnx_data')):
            p = os.path.join(root, f)
            mb = os.path.getsize(p) / 1024**2
            total += mb
            rel = os.path.relpath(p, 'onnx-int8')
            print(f'  {rel:50s} {mb:>8.0f} MB')
print(f'  {"TOTAL":50s} {total:>8.0f} MB ({total/1024:.1f} GB)')

if os.path.exists('output/test_result.png'):
    print(f'\nTest output: {PROJECT_DIR}/output/test_result.png')

# Pack as tar.gz nếu cần copy ra ngoài
print('\n--- Optional: pack for transfer ---')
print(f'tar -czf {WORKDIR}/instantid-int8.tar.gz -C {PROJECT_DIR} onnx-int8/')

print(f'\nDone! All files saved locally in {PROJECT_DIR}/')

---
## Environment Info

| | Colab Free (T4) | **Run.ai H200** |
|---|---|---|
| VRAM | 16 GB | **80 GB** |
| RAM | 12 GB | **200+ GB** |
| UNet Quantize | FAIL (OOM) | **OK** |
| Total Time | ~2 hours | **~30 min** |
| Final Size | ~7 GB (UNet unquantized) | **~2.5 GB** (all INT8) |
| Storage | Google Drive | **Local /workspace** |